In [1]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
from sklearn.preprocessing import StandardScaler
import matplotlib.pyplot as plt

In [2]:
data=pd.read_csv("cbc information.csv") # input data

In [3]:
def ArrayOfVectors(frequency, features):
    array = np.random.rand(frequency, features)
    norm = np.linalg.norm(array, axis = 0)
    normalized = array / norm
    return normalized.T

In [4]:
def median(projection):
    median = np.median(projection, axis = 0)
    return median

In [5]:
def split_data(rows, projection):
    sort_proj = np.sort(projection, axis = 0)
    split_rows = int(rows/2)
    split_1 = sort_proj[:split_rows,:]
    split_2 = sort_proj[split_rows:,:]
    split_2 = split_2[::-1]
    return split_1, split_2

In [6]:
def medcouple(med, projection, rows):
    split_1, split_2 = split_data(rows, projection)
    diff = (split_2 - med) - (med - split_1)
    h = diff / (split_2 - split_1)
    medcouple = np.median(h, axis = 0)
    return medcouple

In [7]:
def Quartiles(projection, mc):
    Q1 = np.percentile(projection, 25, axis = 0).reshape(-1,1)
    Q3 = np.percentile(projection, 75, axis = 0).reshape(-1,1)
    IQR = Q3 - Q1
    lower = np.where( mc < 0, Q1 - 1.5 * np.exp(-3 * mc) * IQR, Q1 - 1.5 * np.exp(-4 * mc) * IQR)
    upper = np.where( mc < 0, Q3 + 1.5 * np.exp(4 * mc) * IQR, Q3 + 1.5 * np.exp(3 * mc) * IQR)
    return lower, upper

In [8]:
def supremum(projection, mc, med):
    w1, w2 = Quartiles(projection, mc)
    w1 = w1.reshape(1,-1)
    w2 = w2.reshape(1,-1)
    AO = np.where(projection > med, (projection - med) / (w2 - med), (med - projection) / (med - w1))
    sup = np.max(AO, axis = 1).reshape(-1,1)
    return sup

In [9]:
def Outliers(sup, rows):
    sup_med = median(sup)
    AO_MC = medcouple(sup_med, sup, rows)
    AO_lower_limit, AO_upper_limit = Quartiles(sup, AO_MC)
    normal_rows, normal_cols = np.where(sup > AO_lower_limit)
    outliers_rows, outlier_cols = np.where(sup > AO_upper_limit)
    return outliers_rows, normal_rows

In [10]:
def main(data):
    features = data.shape[1] # number of features
    frequency = 400 # number of projections
    rows = data.shape[0] # number of examples
    model = StandardScaler() # creating a scaling model
    info = model.fit_transform(data) # fitting the scaling model with data
    # creating an array of unit vectors
    array = ArrayOfVectors(frequency, features)
    # performing dot product with array to get projections
    projection = np.dot(info, array)
    # finding the median of the projection
    med = median(projection).reshape(1,-1)
    # finding the medcouple
    mc = medcouple(med, projection, rows).reshape(-1,1)
    # finding the values of whiskers
    w1, w2 = Quartiles(projection, mc)
    # finding the AO values of each data point
    sup = supremum(projection, mc, med)
    # declaring outliers
    outlier_rows, normal_rows = Outliers(sup, rows)
    # creating a label called "Anomaly"
    labels = np.zeros(rows)
    labels[outlier_rows] = 1
    data["anomaly"] = labels
    

In [11]:
main(data)